# Machine Learning for Geoscience
### CU Boulder — Python for Geologists

You are an exploration geologist working a gold district in Nevada. A regional stream sediment survey has returned **57 element concentrations** at hundreds of sample sites across the study area. Your job: find patterns in that data that point toward buried mineralisation.

Human eyes can compare two elements at a time. ML can compare all 57 simultaneously — and do it across the whole landscape at once.

In this session you will run three techniques on that dataset:

| Method | Type | What it does here |
|--------|------|-------------------|
| **PCA** | Unsupervised | Compresses 57 elements into a handful of summary scores |
| **K-means** | Unsupervised | Groups samples by geochemical similarity — no labels needed |
| **Random Forest** | Supervised | Learns from known deposits to score every sample location |

These are not exploration-specific tools. As you work through each section you will see sidebars showing where the exact same technique appears in stratigraphy, petroleum geology, environmental science, and petrology. The dataset is exploration-flavoured; the skills transfer directly.

**How this notebook works:** Most cells are already written. Cells marked `# --- YOUR TURN ---` ask you to change one or two numbers, re-run the cell, and interpret the output. You cannot break anything — fallback defaults are built in.

---
## Section 0: Setup
*Run these cells once at the start. They install packages (Colab only), import libraries, and point the notebook at the data folder.*

In [ ]:
# SECTION 0: SETUP | ~3 min
# ── Colab only: install missing packages ──────────────────────────────────
import sys, subprocess

COLAB = 'google.colab' in sys.modules
if COLAB:
    print('Installing packages for Colab...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'geopandas', 'rasterio', 'scikit-learn', 'matplotlib', 'numpy', 'pandas', 'scipy'
    ], check=True)
    print('Done.')
else:
    print('Not on Colab — skipping pip install.')

In [ ]:
# Standard imports
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

# Tell the helpers where to find the data
NOTEBOOK_DIR = Path('.')          # adjust if running from a different directory
DATA_DIR     = NOTEBOOK_DIR / 'data'

os.environ['UCB_PROJECT_ROOT'] = str(NOTEBOOK_DIR.resolve())
os.environ['UCB_DATA_ROOT']    = str(DATA_DIR.resolve())

# Load the helper library
import helpers as h

# ── Data paths ─────────────────────────────────────────────────────────────
GEOCHEM_PATH      = DATA_DIR / 'vector' / 'geochem.geojson'
LITHOLOGY_PATH    = DATA_DIR / 'vector' / 'lithology.geojson'
OCCURRENCES_PATH  = DATA_DIR / 'vector' / 'mineral occurrences.geojson'
SPECTRAL_DIR      = DATA_DIR / 'raster' / 'spectral'
PROSPECTIVITY_TIF = DATA_DIR / 'ML' / 'stacked_result.tif'

print('Data directory:', DATA_DIR.resolve())
print('Paths configured.')

---
## Section 1: Why ML for Geology?

### The exploration problem

A stream sediment survey in Nevada returns 57 element concentrations per sample. Some of those elements are elevated near gold deposits; most aren't. Some combinations of elements are more informative than any single element alone. And the signal is noisy — natural geochemical variability, sample dilution, and laboratory error all blur the picture.

The classic approach is to plot element pairs one at a time and look for anomalies by eye. That works when you have 3 elements. With 57, there are 1,596 possible pairwise plots. ML processes all of them simultaneously and surfaces the patterns that matter.

### The same problem appears everywhere in geology

The details differ, but the core challenge is identical across subfields: **too many correlated variables, not enough human eyes**.

- A petroleum geologist looking at 15 well log curves across 200 wells faces the same dimensionality problem
- A stratigrapher working with 40-element geochemistry down a drill core needs to define unit boundaries without subjective bias
- An environmental geologist with 30-element groundwater data across a contaminated site needs to fingerprint which industrial source is responsible
- A petrologist with EPMA mineral chemistry data on 500 grains needs to discriminate magmatic suites

The tools you will use today — PCA, K-means, and Random Forest — were not invented for geology. They are general-purpose. We happen to have a Nevada gold dataset in front of us, but the workflow is the same regardless of what the columns represent.

### Supervised vs unsupervised

**Unsupervised ML** (PCA, K-means): no labels needed. The algorithm finds structure in the data on its own — like sorting coins by size and colour without knowing their value.

**Supervised ML** (Random Forest): you provide labelled examples of what you're looking for — known deposits, identified facies, confirmed contamination sources — and the algorithm learns what distinguishes those from everything else.

> **Instructor note:** The supervised/unsupervised distinction is the key conceptual takeaway. Return to it after each section and ask students which type they just ran.

---
## Section 2: Look at the Data
*Before fitting any model, always look at what you have.*

In [ ]:
# SECTION 2: DATA LOOK | ~5 min | Target: min 10

# Load the datasets
geochem_gdf = h.ensure_xy(gpd.read_file(GEOCHEM_PATH))
vector_gdf  = gpd.read_file(LITHOLOGY_PATH)
targets_gdf = gpd.read_file(OCCURRENCES_PATH)

# Feature columns = numeric, not coordinates
EXCLUDE = {'X', 'Y', 'id', 'coord_x', 'coord_y', 'elevation_m', 'OBJECTID', 'FID'}
feature_cols = [
    c for c in geochem_gdf.select_dtypes(include=[np.number]).columns
    if c not in EXCLUDE
]

print(f"Geochem samples   : {len(geochem_gdf)}")
print(f"Lithology units   : {len(vector_gdf)}")
print(f"Known occurrences : {len(targets_gdf)}")
print(f"\nGeochem element columns ({len(feature_cols)} total):")
print(feature_cols[:10], '...')

In [ ]:
# Quick table view — what does each row look like?
geochem_gdf[feature_cols[:8]].head()

In [ ]:
# Map: sample locations on lithology, with known deposit locations marked
fig, ax = plt.subplots(figsize=(10, 8))

h.plot_vector(vector_gdf, ax=ax, title='Sample Locations and Known Deposits',
              alpha=0.3, edgecolor='grey', linewidth=0.4)

geochem_gdf.plot(ax=ax, color='steelblue', markersize=15, alpha=0.7,
                 label='Geochem samples')

tgt_proj = targets_gdf.to_crs(geochem_gdf.crs) if targets_gdf.crs != geochem_gdf.crs else targets_gdf
tgt_proj.plot(ax=ax, marker='*', color='gold', markersize=180,
              edgecolor='black', linewidth=0.8, label='Known deposits', zorder=5)

ax.legend()
plt.tight_layout()
plt.show()

**What you're looking at:** Blue dots are geochemical sample sites. Gold stars are known mineral occurrences. Background polygons are mapped lithology units.

**Interpret this map:**
1. Are sample sites evenly distributed, or do some areas have denser coverage?
2. Do any occurrences cluster in a particular lithology?
3. What does the spatial distribution of samples tell you about potential data bias?

> **Instructor note:** Spatial sampling bias is a real problem in any field application — the same issue appears in facies classification from wells (wells are drilled where you expect something, not at random).

---
## Section 3: PCA — Compressing 57 Elements Into Something Useful

### What is PCA?

In our Nevada dataset, many elements move together. Cu, Mo, and Au co-vary in porphyry systems. As, Sb, and Hg co-vary along epithermal pathways. Fe and Mn track weathering. PCA finds those co-varying groups and replaces them with new summary variables called **principal components**.

Instead of 57 maps — one per element — you get 5 summary maps that together capture most of the geochemical information. PC1 captures the dominant pattern in the dataset; PC2 captures the next largest independent pattern; and so on.

**What is variance explained?** Each PC explains some fraction of the total spread in the data. If PC1 explains 40% of variance, then 40% of all geochemical variation across the study area is described by that single new axis.

---

> **The same idea in other fields:**
> - *Stratigraphy:* PCA of drill core geochemistry (30+ elements) collapses the data into 2–3 scores that reveal chemostratigraphic packages — useful when the visual lithology is monotonous shale
> - *Petrology:* Major element PCA of volcanic rocks automatically recovers the classic TAS and AFM discrimination diagrams from the data, without manually choosing axes
> - *Environmental:* Multi-element groundwater PCA separates natural background from contamination plumes, and can distinguish multiple industrial sources by their loading signatures
> - *Petroleum:* PCA of log suites (GR, density, sonic, resistivity) creates composite curves that reduce noise and correlate better across wells than any single log

---

> **Instructor note:** The coin-sorting analogy works well: sorting by size first, then by colour, is analogous to PCA finding the axes of maximum variation in order of importance.

In [ ]:
# SECTION 3: PCA | ~15 min | Target: min 10

# --- YOUR TURN ---
# Should we log-transform the geochemistry before PCA?
# Geochemical data is log-normally distributed — a few very high outliers dominate
# the variance if you don't transform first.
# 'log1p' applies log(x+1) to compress those outliers.
# Try 'none' and compare the variance plot — does PC1 dominate more without the transform?
#
TRANSFORM = 'log1p'   # Options: 'log1p'  or  'none'
SCALE = True          # Options: True (standardise each element) or False

# Safety fallback — don't edit below this line
try:
    assert TRANSFORM in ('log1p', 'none'), "TRANSFORM must be 'log1p' or 'none'"
    assert isinstance(SCALE, bool), "SCALE must be True or False"
except AssertionError as e:
    print(f'Invalid setting: {e}  → using defaults')
    TRANSFORM, SCALE = 'log1p', True

pca_inputs = h.prepare_pca_inputs(
    geochem_gdf, feature_cols,
    transform=TRANSFORM,
    scale_features=SCALE,
)
X_scaled = pca_inputs['X_scaled']
pca_cols  = pca_inputs['pca_cols']
print(f'Transform: {TRANSFORM} | Scale: {SCALE}')
print(f'Feature matrix shape: {X_scaled.shape}')

In [ ]:
# Fit PCA — keep all components so we can see the full variance curve
pca, X_pca = h.fit_pca_model(X_scaled)
print(f'Input dimensions : {X_scaled.shape[1]} elements')
print(f'Output dimensions: {X_pca.shape[1]} principal components')
print(f'Shape of PCA scores: {X_pca.shape}  (one row per sample)')

In [ ]:
# Variance explained plot — how many PCs do we actually need?
fig, ax = h.plot_pca_variance(pca)
plt.show()

**What you're looking at:** Each bar is one principal component. The red line shows cumulative variance. The dashed lines mark 50%, 75%, and 90% thresholds.

**Interpret this plot:**
1. PC1 explains ____% of variance.
2. How many PCs do you need to reach 75% of total variance? _____
3. Where is the "elbow" — the point where adding another PC stops helping much? _____
4. If you used `TRANSFORM = 'none'`, how does PC1% change? What does that tell you about raw geochemical data distributions?

In [ ]:
# --- YOUR TURN ---
# How many PCA components should we plot loadings for?
# A "loading" tells you which variables drive each PC.
# Positive loading = variable increases with PC score.
# Negative loading = variable decreases with PC score.
# Try 2, 3, or 5 — do later PCs still make geological sense?
#
N_LOAD_COMPONENTS = ___   # Try: 2, 3, or 5

# Safety fallback
try:
    N_LOAD_COMPONENTS = int(N_LOAD_COMPONENTS)
    assert 1 <= N_LOAD_COMPONENTS <= pca.n_components_, \
        f'Must be between 1 and {pca.n_components_}'
except Exception as e:
    print(f'Invalid input: {e}  → using N_LOAD_COMPONENTS = 3')
    N_LOAD_COMPONENTS = 3

fig, axes = h.plot_pca_loadings(pca, pca_cols, n_components=N_LOAD_COMPONENTS)
plt.show()

**What you're looking at:** Each subplot shows one PC. Green bars load positively; orange bars load negatively. Bar length shows how strongly that element contributes.

**Interpret these loadings:**
1. What element has the strongest positive loading on PC1? _____
2. What element has the strongest negative loading on PC1? _____
3. Elements like As, Sb, Hg, and Tl are classic hydrothermal pathfinders. Which PC are they associated with?
4. What geological process might explain the pattern you see in PC2?

> **Instructor note:** If time allows, ask students to hypothesise what a PC loading pattern would look like in their own field — e.g., what would PC1 look like for a well log PCA in a carbonate succession?

In [ ]:
# Map the first 3 PCA scores in geographic space
fig, axes = h.plot_spatial_pca_components(
    geochem_gdf, X_pca, pca, n_components=3
)
plt.show()

**What you're looking at:** Each map shows the PC score at each sample location. Warm = high; cool = low. Compare these to the lithology map from Section 2.

**Interpret these maps:**
1. Does PC1 vary smoothly across the study area, or does it show sharp boundaries?
2. Do PC score boundaries correspond to lithology contacts? What would that mean?
3. Do any known deposit locations appear in high or low PC score zones?
4. If this were a chemostratigraphy dataset (core samples down a section instead of points across a map), what would each map represent?

---
## Section 4: K-means — Finding Geochemical Populations

### What is K-means?

PCA compressed our 57 elements into summary scores. K-means takes the next step: it partitions every sample into one of *k* discrete groups based on similarity across all variables at once.

In an exploration context, those groups might represent distinct geochemical populations — background granites, altered zones, mineralised samples. But we don't tell the algorithm that. It finds the groupings from the data alone, and then we interpret them geologically.

Unlike PCA, K-means gives each sample a **label** (Cluster 0, Cluster 1, ...) that can be plotted directly on a map and compared against lithology, structure, or deposit locations.

**The challenge:** you have to choose *k* before running. We will use two diagnostics to help:
- **Elbow plot:** how much does within-cluster variance decrease as k increases? Look for the bend.
- **Silhouette score:** how cleanly separated are the clusters? Higher is better (max = 1).

---

> **The same idea in other fields:**
> - *Petroleum geology:* K-means on GR, density, and sonic logs classifies every depth interval into electrofacies — widely used when no core is available and the geologist needs discrete units for reservoir modelling
> - *Stratigraphy:* Clustering element ratios down a drill core reveals chemostratigraphic boundaries without relying on visual description, which is inherently subjective
> - *Provenance:* Detrital zircon or heavy mineral chemistry clustered to identify how many source regions contributed to a sedimentary basin, and in what proportions
> - *Environmental:* Groundwater samples clustered to separate pristine aquifer from contaminated zones and mixing end-members — useful before any regulatory sampling decisions

---

> **Instructor note:** There is no single correct k. Two geologists might reasonably choose different values and both be right — the goal is geological interpretability, not metric optimisation.

In [ ]:
# SECTION 4: K-MEANS | ~13 min | Target: min 10

# Run diagnostics for k = 2 through 8
K_RANGE = range(2, 9)
print('Running K-means diagnostics for k =', list(K_RANGE), '...')
inertias, silhouettes = h.run_kmeans_diagnostics(X_scaled, K_RANGE)
print('Done.')

In [ ]:
# Elbow and silhouette plots
fig, axes = h.plot_elbow_silhouette(list(K_RANGE), inertias, silhouettes)
plt.show()

**What you're looking at:** Left = elbow plot (lower inertia = tighter clusters). Right = silhouette scores (higher = more distinct clusters; 0.2–0.5 is typical for real geochemical data).

**Interpret these plots:**
1. Where is the elbow in the left plot? _____
2. Which k gives the highest silhouette score? _____
3. Do the two diagnostics agree? If not, which would you trust more, and why?
4. What would k = 2 represent geologically (e.g., in a porphyry system? in a shelf carbonate succession)?

In [ ]:
# --- YOUR TURN ---
# Choose the number of clusters based on the plots above.
# The algorithm will suggest an elbow-based value, but you can override it.
# Try at least two different values and compare the maps that follow.
#
MY_K = ___   # Try: 3, 4, or 5

# Safety fallback
try:
    MY_K = int(MY_K)
    assert 2 <= MY_K <= 10, 'k must be between 2 and 10'
except Exception as e:
    print(f'Invalid k: {e}  → using k = 4')
    MY_K = 4

chosen_k       = h.choose_k(list(K_RANGE), inertias, user_override=MY_K)
cluster_labels = h.run_kmeans_clustering(X_scaled, n_clusters=chosen_k)

print(f'\nCluster label counts:')
unique, counts = np.unique(cluster_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Cluster {u}: {c} samples')

In [ ]:
# PCA scatter coloured by cluster — do the groups separate cleanly in PC space?
fig, ax = h.plot_kmeans_pca_scatter(X_pca, cluster_labels)
plt.show()

In [ ]:
# Geographic overlay — do clusters follow lithology contacts?
fig, ax = h.plot_clusters_on_lithology(vector_gdf, geochem_gdf, cluster_labels)
plt.show()

**What you're looking at:** Clusters overlaid on lithology polygons. If clusters align with lithology contacts, the element patterns are controlled by source rock composition. If they cut across contacts, something else is driving them — possibly hydrothermal alteration, or a transported signal.

**Interpret these maps:**
1. Do any clusters follow lithology contact lines? _____
2. Are any clusters scattered across multiple lithologies? What might that indicate?
3. Are known deposit locations concentrated in one cluster?
4. Try a different MY_K. Do the same groupings persist at different k, or do boundaries shift dramatically?

> **Instructor note:** Clusters that track lithology = lithogeochemistry (source rock control). Clusters that cut contacts = hydrothermal overprinting. Same logic applies to well log electrofacies — do they match the stratigrapher's core description?

---
## Section 5: Supervised ML — Predicting Mineral Prospectivity

### From patterns to predictions

PCA and K-means found structure in the data without any labels. Now we give the algorithm something specific to learn: **known mineral occurrences**.

We label each sample location as:
- **1 (positive)** — within 500 m of a known deposit
- **0 (background)** — everything else

Then we train a **Random Forest**: an ensemble of decision trees, each trained on a random subset of the data. Each tree votes on whether a new sample looks deposit-like. The output is a probability from 0 to 1.

```
Spectral bands  ──►  Feature matrix  ──►  Random Forest  ──►  Probability score
(at sample pts)      (standardised)       (100 trees)         (0 = background, 1 = deposit-like)
```

**Features for this section:** We extract spectral remote sensing values at each sample location. This avoids any spatial interpolation — each sample point simply reads the pixel underneath it from the satellite-derived raster.

> **Instructor note:** The pre-generated prospectivity map at the end was built by applying this same method to every pixel in the study area — with geochem, spectral, and geophysics all combined. What students train here is the same algorithm with one data type.

In [ ]:
# SECTION 5: SUPERVISED ML | ~20 min | Target: min 10

# ── Step 1: Extract spectral values at each geochem sample location ────────
import rasterio
from pyproj import Transformer

spec_paths = sorted(SPECTRAL_DIR.glob('*.tif'))
if not spec_paths:
    raise FileNotFoundError(f'No .tif files found in {SPECTRAL_DIR}')

spectral_bands = {}
for path in spec_paths:
    with rasterio.open(path) as src:
        # Re-project sample coordinates to raster CRS if needed
        if src.crs and geochem_gdf.crs and src.crs != geochem_gdf.crs:
            t = Transformer.from_crs(geochem_gdf.crs, src.crs, always_xy=True)
            xs, ys = t.transform(
                geochem_gdf.geometry.x.values,
                geochem_gdf.geometry.y.values
            )
        else:
            xs = geochem_gdf.geometry.x.values
            ys = geochem_gdf.geometry.y.values

        coords = list(zip(xs, ys))
        vals = np.array([v[0] for v in src.sample(coords)], dtype=float)
        nodata = src.nodata
        if nodata is not None:
            vals[vals == nodata] = np.nan
        spectral_bands[path.stem] = vals

spec_df   = pd.DataFrame(spectral_bands)
spec_names = list(spectral_bands.keys())

print(f'Spectral bands extracted: {len(spec_names)}')
print(f'Band names: {spec_names}')
print(f'Missing values (outside raster): {spec_df.isnull().sum().sum()}')

In [ ]:
# ── Step 2: Label each sample point ───────────────────────────────────────
y_labels = h.prepare_ml_labels(geochem_gdf, targets_gdf, radius_m=500)

n_pos = int(y_labels.sum())
n_neg = len(y_labels) - n_pos
print(f'\nPositive : {n_pos}  ({100*n_pos/len(y_labels):.1f}%)')
print(f'Background: {n_neg}  ({100*n_neg/len(y_labels):.1f}%)')
print("\nNote: strong imbalance is normal — deposits are rare.")
print("We'll use class_weight='balanced' to handle this.")

# Map of training labels
fig, ax = plt.subplots(figsize=(10, 7))
h.plot_vector(vector_gdf, ax=ax, alpha=0.2, edgecolor='grey', linewidth=0.4)
geochem_gdf[y_labels == 0].plot(ax=ax, color='steelblue', markersize=12,
                                 alpha=0.6, label='Background')
geochem_gdf[y_labels == 1].plot(ax=ax, color='red', markersize=25,
                                 alpha=0.9, label='Near deposit (positive)')
tgt_proj.plot(ax=ax, marker='*', color='gold', markersize=180,
              edgecolor='black', linewidth=0.8, label='Known deposits', zorder=5)
ax.set_title('Training Labels')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 3: Build feature matrix from spectral bands ──────────────────────
# Drop rows where any spectral band is missing (sample outside raster coverage)
valid_mask = spec_df.notnull().all(axis=1).values

X_spec_raw = spec_df.values[valid_mask]
y_valid    = y_labels[valid_mask]
geochem_valid = geochem_gdf.iloc[valid_mask].reset_index(drop=True)

# Standardise so bands with different value ranges contribute equally
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_ml   = scaler.fit_transform(X_spec_raw)

print(f'Samples with full spectral coverage: {valid_mask.sum()} / {len(valid_mask)}')
print(f'Feature matrix: {X_ml.shape[0]} samples × {X_ml.shape[1]} spectral bands')

In [ ]:
# --- YOUR TURN ---
# Choose settings for the Random Forest.
# N_ESTIMATORS = number of decision trees. More = slower but often more stable.
# MAX_DEPTH    = how deep each tree can grow. None = unlimited.
#
# Uncomment one option for each, re-run, and compare the ROC curve AUC.

N_ESTIMATORS = 100    # options:  50 (fast)  |  100 (balanced)  |  200 (thorough)
# N_ESTIMATORS = 50
# N_ESTIMATORS = 200

MAX_DEPTH = None      # options:  None (full depth)  |  5 (shallow)  |  10 (medium)
# MAX_DEPTH = 5
# MAX_DEPTH = 10

# Safety fallback
try:
    N_ESTIMATORS = int(N_ESTIMATORS)
    assert 10 <= N_ESTIMATORS <= 500
    if MAX_DEPTH is not None:
        MAX_DEPTH = int(MAX_DEPTH)
        assert 1 <= MAX_DEPTH <= 50
except Exception as e:
    print(f'Invalid setting: {e}  → using defaults')
    N_ESTIMATORS, MAX_DEPTH = 100, None

print(f'Will train: {N_ESTIMATORS} trees, max depth = {MAX_DEPTH}')

In [ ]:
# Train and evaluate
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_ml, y_valid, test_size=0.25, random_state=42, stratify=y_valid
)
print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples')

rf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

y_prob = rf.predict_proba(X_test)[:, 1]
print(f'\nModel trained. Predicted probability range: {y_prob.min():.3f} – {y_prob.max():.3f}')

importance_df = pd.DataFrame({
    'feature'   : spec_names,
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False)

In [ ]:
# ROC and Precision-Recall curves
fig, axes = h.plot_roc_pr_curves(y_test, y_prob)
plt.show()

**What you're looking at:** The ROC curve shows the trade-off between catching real deposits (True Positive Rate) and generating false alarms (False Positive Rate). The area under the curve (AUC) is the headline number.

**Interpret these curves:**
1. What is the AUC of your model? _____
2. AUC = 0.5 means _____ . AUC = 1.0 means _____.
3. In mineral exploration, false negatives (missing a real deposit) are usually more costly than false positives (drilling a dud). Does that change where you'd set the threshold on the PR curve?
4. Try `N_ESTIMATORS = 50` vs `200`. Does AUC change meaningfully?

> **Instructor note:** Spectral-only AUC will vary depending on the dataset. The full pipeline (geochem + spectral + geophysics) typically improves AUC because each data type contributes complementary information.

In [ ]:
# --- YOUR TURN ---
# How many top spectral bands do you want to display?
# The bar chart shows which bands the Random Forest found most informative.
#
TOP_N = ___   # Try: 3, 5, or all (set to len(spec_names))

# Safety fallback
try:
    TOP_N = int(TOP_N)
    assert 1 <= TOP_N <= len(spec_names), f'Must be between 1 and {len(spec_names)}'
except Exception as e:
    print(f'Invalid input: {e}  → using TOP_N = {min(6, len(spec_names))}')
    TOP_N = min(6, len(spec_names))

fig, ax = h.plot_feature_importance(importance_df, top_n=TOP_N)
plt.show()

**What you're looking at:** Which spectral bands drove the model's decisions. Longer bar = more important. These importances reflect predictive power, not physical correlation — a band that distinguishes deposits from background will score high even if its absolute values are small.

**Interpret this chart:**
1. What is the most important spectral band? _____
2. Spectral indices like clay hydroxyl and iron oxide bands are associated with hydrothermal alteration. Are any of those near the top?
3. If you set `MAX_DEPTH = 5`, does the importance ranking change? Why might a shallower tree use different bands?
4. What would you expect feature importances to look like if you used all three data types (geochem + spectral + geophysics)?

In [ ]:
# Load and display the pre-generated prospectivity map.
# This was produced by the full pipeline applied to every pixel:
# geochem (interpolated) + spectral + geophysics, combined across
# multiple independent model runs. What you just trained is the same method.

with rasterio.open(PROSPECTIVITY_TIF) as src:
    prob_map = src.read(1).astype(float)
    nodata   = src.nodata
    if nodata is not None:
        prob_map[prob_map == nodata] = np.nan
    extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

print(f'Prospectivity map shape: {prob_map.shape}')
print(f'Value range: {np.nanmin(prob_map):.3f} – {np.nanmax(prob_map):.3f}')

fig, axes = h.plot_prospectivity_map(prob_map, extent=extent, threshold=0.5)
plt.suptitle(
    'Pre-generated Prospectivity Map\n'
    '(full pipeline: geochem + spectral + geophysics, applied to every pixel)',
    fontsize=11
)
plt.tight_layout()
plt.show()

print()
print('This map was produced by the same Random Forest method you just ran,')
print('with more input features and a spatial cross-validation scheme.')

**What you're looking at:** Left = continuous probability map (0–1). Right = binary classification at 0.5 threshold. Green = prospective.

**Interpret this map:**
1. Do high-probability zones overlap with the known deposits?
2. Are there green zones with no known deposits? Are these model errors, or candidates for new drilling?
3. How does the spatial pattern compare to the cluster map from Section 4?
4. The threshold is 0.5. Would you lower it (more ground covered, more false alarms) or raise it (fewer targets, lower risk of missing deposits)? What drives that decision?

> **Instructor note:** This is the deliverable. The drilling decision still belongs to the geologist — the map provides a ranked list of targets, not a guarantee. The same logic applies to any supervised ML product: it ranks options, it doesn't make choices.

---
## Section 6: Wrap-Up

### What you covered today

| Section | Method | Type | What it produced |
|---------|--------|------|------------------|
| 3 | PCA | Unsupervised | Summary geochemical components; element association patterns |
| 4 | K-means | Unsupervised | Discrete geochemical populations mapped against lithology |
| 5 | Random Forest | Supervised | Deposit probability score at each sample location |

### The exploration geologist's toolkit — and everyone else's

You ran all three techniques on a Nevada gold dataset, but none of them know they're doing exploration. Swap the input data and the interpretation changes; the code does not.

| Your data | PCA gives you | K-means gives you | Supervised ML gives you |
|-----------|--------------|-------------------|------------------------|
| Well log suite | Composite curves, noise-reduced | Electrofacies without core | Reservoir quality prediction |
| Core geochemistry | Chemostratigraphic axes | Unit boundaries, objective picks | Lithology prediction at unsampled depths |
| Groundwater chemistry | Source fingerprints | Contamination populations | Risk scores at unmonitored wells |
| Satellite multispectral | Spectral end-members | Land cover / alteration classes | Hazard or resource probability maps |

### How the full exploration pipeline works

The notebook trained on point data — one row per sample location. In production:
- Geochem, spectral, and geophysical rasters are all combined as input features
- A spatial cross-validation scheme prevents the model from memorising deposit locations instead of learning the underlying signal
- Multiple independent model runs are stacked — the final map is a consensus probability
- Uncertainty is quantified alongside the probability score and delivered to the client

The core ML is what you ran today. The complexity is in the data pipeline around it.

### Topics to explore next

- **Anomaly detection** with Isolation Forest — flags unusual samples without any deposit labels
- **Positive-unlabelled learning** — formally handles the fact that "background" ≠ "confirmed absence of deposit"
- **Spatial cross-validation** — why random train/test split over-estimates model performance for spatially correlated data
- **UMAP / t-SNE** — non-linear dimensionality reduction for datasets where PCA misses curved structure